# ARC-AGI-2 Object/Relation Search v2

v1 established two things:

* grid-program search has substantial behavioral redundancy;
* the current language has **zero demonstration coverage** on the smoke-test evaluation tasks.

So v2 changes the representation rather than the ranking rule.

### Research questions

1. **Coverage:** does an object/relation-oriented candidate language exactly explain more ARC tasks?
2. **Compute:** at the *same DSL and search depth*, does behavioral quotienting reduce candidate generation / candidate-input executions?
3. **Probe safety:** do task-conditioned counterfactual probes preserve more behaviorally distinct hypotheses than demonstration-only quotienting?

### Evaluation hygiene

Development is done on a deterministic split of the **1,000 training tasks**. The official 120-task evaluation set is disabled by default and should be touched only at frozen milestones.

The quotient relation remains operational: two programs are equivalent when their output fingerprints agree on a chosen finite probe set. It is **not** claimed to be universal semantic equivalence.


In [ ]:
from __future__ import annotations
import json, time, math, hashlib
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict, Counter, deque
from typing import Callable
import numpy as np, pandas as pd

# ---------- Kaggle data discovery ----------
REQ = {
    "arc-agi_training_challenges.json",
    "arc-agi_training_solutions.json",
    "arc-agi_evaluation_challenges.json",
    "arc-agi_evaluation_solutions.json",
    "arc-agi_test_challenges.json",
}
def data_dir():
    roots=[Path("/kaggle/input/competitions"),Path("/kaggle/input"),Path(".")]
    hits=[]
    for root in roots:
        if not root.exists(): continue
        for f in root.rglob("arc-agi_test_challenges.json"):
            names={p.name for p in f.parent.iterdir()}
            if REQ.issubset(names): hits.append(f.parent)
    if not hits:
        raise FileNotFoundError(
            "Attach 'ARC Prize 2026 - ARC-AGI-2' as Kaggle input, then rerun."
        )
    return sorted(hits,key=lambda p:(0 if "competitions" in str(p) else 1,len(str(p))))[0]

D=data_dir()
load=lambda n: json.load(open(D/n))
TR=load("arc-agi_training_challenges.json")
TS=load("arc-agi_training_solutions.json")
EV=load("arc-agi_evaluation_challenges.json")
ES=load("arc-agi_evaluation_solutions.json")
TE=load("arc-agi_test_challenges.json")
print("ARC data:",D)
print("training/evaluation/test:",len(TR),len(EV),len(TE))

# deterministic 80/20 split, stable across runs
ids=sorted(TR)
DEV_IDS=[t for t in ids if int(hashlib.sha256(t.encode()).hexdigest(),16)%5 != 0]
HELD_IDS=[t for t in ids if t not in set(DEV_IDS)]
print("development:",len(DEV_IDS),"held-out training:",len(HELD_IDS))


In [ ]:
# ---------- grids, objects, relations ----------
def A(g): return np.asarray(g,dtype=np.int8)
def K(x): return tuple(map(tuple,A(x).tolist()))
def bg(x):
    x=A(x); v,c=np.unique(x,return_counts=True)
    return int(v[np.argmax(c)])

def bbox_pts(pts):
    rs=[p[0] for p in pts]; cs=[p[1] for p in pts]
    return min(rs),min(cs),max(rs),max(cs)

@dataclass(frozen=True)
class Obj:
    pts: tuple
    color: int
    size: int
    bbox: tuple
    h: int
    w: int
    holes: int

def count_holes(mask):
    H,W=mask.shape
    seen=np.zeros_like(mask,bool)
    q=deque()
    for r in range(H):
        for c in (0,W-1):
            if not mask[r,c] and not seen[r,c]: seen[r,c]=1; q.append((r,c))
    for c in range(W):
        for r in (0,H-1):
            if not mask[r,c] and not seen[r,c]: seen[r,c]=1; q.append((r,c))
    for r,c in q:
        for dr,dc in ((1,0),(-1,0),(0,1),(0,-1)):
            rr,cc=r+dr,c+dc
            if 0<=rr<H and 0<=cc<W and not mask[rr,cc] and not seen[rr,cc]:
                seen[rr,cc]=1; q.append((rr,cc))
    unseen=(~mask)&(~seen)
    holes=0
    seen2=np.zeros_like(mask,bool)
    for r in range(H):
        for c in range(W):
            if unseen[r,c] and not seen2[r,c]:
                holes+=1; qq=[(r,c)]; seen2[r,c]=1
                for rr,cc in qq:
                    for dr,dc in ((1,0),(-1,0),(0,1),(0,-1)):
                        nr,nc=rr+dr,cc+dc
                        if 0<=nr<H and 0<=nc<W and unseen[nr,nc] and not seen2[nr,nc]:
                            seen2[nr,nc]=1; qq.append((nr,nc))
    return holes

def objects(x, same_color=True, diag=False):
    x=A(x); b=bg(x); H,W=x.shape
    seen=np.zeros((H,W),bool); out=[]
    dirs=[(1,0),(-1,0),(0,1),(0,-1)]
    if diag: dirs += [(1,1),(1,-1),(-1,1),(-1,-1)]
    for r in range(H):
        for c in range(W):
            if seen[r,c] or x[r,c]==b: continue
            col=int(x[r,c]); seen[r,c]=1; q=[(r,c)]; pts=[]
            for rr,cc in q:
                pts.append((rr,cc))
                for dr,dc in dirs:
                    nr,nc=rr+dr,cc+dc
                    if not (0<=nr<H and 0<=nc<W) or seen[nr,nc] or x[nr,nc]==b: continue
                    if same_color and int(x[nr,nc])!=col: continue
                    seen[nr,nc]=1; q.append((nr,nc))
            r0,c0,r1,c1=bbox_pts(pts)
            mask=np.zeros((r1-r0+1,c1-c0+1),bool)
            for rr,cc in pts: mask[rr-r0,cc-c0]=1
            out.append(Obj(tuple(pts),col,len(pts),(r0,c0,r1,c1),
                           r1-r0+1,c1-c0+1,count_holes(mask)))
    return out

def render_obj(x,o,crop=True,color=None):
    x=A(x); b=bg(x); col=o.color if color is None else int(color)
    if crop:
        r0,c0,r1,c1=o.bbox
        y=np.full((r1-r0+1,c1-c0+1),b,dtype=np.int8)
        for r,c in o.pts:y[r-r0,c-c0]=col
    else:
        y=np.full_like(x,b)
        for r,c in o.pts:y[r,c]=col
    return y

def crop_nonbg(x):
    x=A(x); pts=np.argwhere(x!=bg(x))
    if not len(pts): return None
    lo,hi=pts.min(0),pts.max(0)
    return x[lo[0]:hi[0]+1,lo[1]:hi[1]+1].copy()

def pick_obj(x, selector, diag=False):
    os=objects(x,True,diag)
    if not os:return None
    if selector=="largest": return max(os,key=lambda o:(o.size,-o.bbox[0],-o.bbox[1]))
    if selector=="smallest": return min(os,key=lambda o:(o.size,o.bbox[0],o.bbox[1]))
    if selector=="top": return min(os,key=lambda o:(o.bbox[0],o.bbox[1]))
    if selector=="bottom": return max(os,key=lambda o:(o.bbox[2],-o.bbox[1]))
    if selector=="left": return min(os,key=lambda o:(o.bbox[1],o.bbox[0]))
    if selector=="right": return max(os,key=lambda o:(o.bbox[3],-o.bbox[0]))
    if selector=="most_holes": return max(os,key=lambda o:(o.holes,o.size))
    if selector=="unique_size":
        cnt=Counter(o.size for o in os)
        u=[o for o in os if cnt[o.size]==1]
        return u[0] if len(u)==1 else None
    if selector=="unique_color":
        cnt=Counter(o.color for o in os)
        u=[o for o in os if cnt[o.color]==1]
        return u[0] if len(u)==1 else None
    return None

def fill_holes(x):
    x=A(x); y=x.copy(); changed=False
    for o in objects(x,True,False):
        r0,c0,r1,c1=o.bbox
        mask=np.zeros((o.h,o.w),bool)
        for r,c in o.pts: mask[r-r0,c-c0]=1
        H,W=mask.shape; seen=np.zeros_like(mask,bool); q=deque()
        for r in range(H):
            for c in (0,W-1):
                if not mask[r,c] and not seen[r,c]:seen[r,c]=1;q.append((r,c))
        for c in range(W):
            for r in (0,H-1):
                if not mask[r,c] and not seen[r,c]:seen[r,c]=1;q.append((r,c))
        for r,c in q:
            for dr,dc in ((1,0),(-1,0),(0,1),(0,-1)):
                rr,cc=r+dr,c+dc
                if 0<=rr<H and 0<=cc<W and not mask[rr,cc] and not seen[rr,cc]:
                    seen[rr,cc]=1;q.append((rr,cc))
        holes=(~mask)&(~seen)
        if holes.any():
            rr,cc=np.where(holes); y[rr+r0,cc+c0]=o.color; changed=True
    return y if changed else None

def bbox_draw(x,mode):
    x=A(x); b=bg(x); pts=np.argwhere(x!=b)
    if not len(pts):return None
    lo,hi=pts.min(0),pts.max(0); r0,c0=lo; r1,c1=hi
    col=Counter(map(int,x[x!=b])).most_common(1)[0][0]
    y=x.copy() if mode.startswith("on_") else np.full_like(x,b)
    if mode.endswith("fill"): y[r0:r1+1,c0:c1+1]=col
    else:
        y[r0,c0:c1+1]=col;y[r1,c0:c1+1]=col
        y[r0:r1+1,c0]=col;y[r0:r1+1,c1]=col
    return y

def split_panels(x):
    x=A(x); H,W=x.shape
    for r in range(1,H-1):
        if len(set(map(int,x[r])))==1 and x[:r].shape==x[r+1:].shape:
            return x[:r],x[r+1:]
    for c in range(1,W-1):
        if len(set(map(int,x[:,c])))==1 and x[:,:c].shape==x[:,c+1:].shape:
            return x[:,:c],x[:,c+1:]
    return None

def panel_logic(x,op):
    z=split_panels(x)
    if z is None:return None
    a,b=z; ma=a!=bg(a); mb=b!=bg(b)
    if op=="union":m=ma|mb
    elif op=="inter":m=ma&mb
    elif op=="xor":m=ma^mb
    elif op=="a-b":m=ma&~mb
    else:m=mb&~ma
    return m.astype(np.int8)

def complete_sym(x,kind):
    x=A(x); b=bg(x)
    if kind=="lr": z=np.fliplr(x)
    elif kind=="ud": z=np.flipud(x)
    elif kind=="rot": z=np.rot90(x,2)
    else:return None
    y=x.copy(); m=(y==b)&(z!=b); y[m]=z[m]
    return y


In [ ]:
# ---------- programs and task-fitted primitives ----------
@dataclass
class Program:
    name:str
    cost:float
    fn:Callable
    def __call__(self,g):
        try:
            y=self.fn(A(g))
            if y is None:return None
            y=A(y)
            if y.ndim!=2 or y.size==0 or max(y.shape)>30:return None
            if y.min()<0 or y.max()>9:return None
            return y
        except Exception:
            return None

def compose(p,q):
    def f(x):
        y=p(x)
        return None if y is None else q(y)
    return Program(f"{q.name}@{p.name}",p.cost+q.cost+.25,f)

def infer_global_color_map(train,p):
    mp={}
    for ex in train:
        u=p(ex["input"]); v=A(ex["output"])
        if u is None or u.shape!=v.shape:return None
        for a,b in zip(u.flat,v.flat):
            a,b=int(a),int(b)
            if a in mp and mp[a]!=b:return None
            mp[a]=b
    if all(a==b for a,b in mp.items()):return None
    def f(x):
        u=p(x)
        if u is None:return None
        y=u.copy()
        for a,b in mp.items():y[u==a]=b
        return y
    return Program(p.name+"+cmap"+str(sorted(mp.items())),p.cost+1+.15*len(mp),f)

def scale_candidates(task):
    pairs=[(A(e["input"]).shape,A(e["output"]).shape) for e in task["train"]]
    ps=[]
    for kr in range(2,7):
        for kc in range(2,7):
            if all(oh==ih*kr and ow==iw*kc for (ih,iw),(oh,ow) in pairs):
                ps.append(Program(f"zoom_{kr}x{kc}",3,lambda x,kr=kr,kc=kc: np.repeat(np.repeat(x,kr,0),kc,1)))
                ps.append(Program(f"tile_{kr}x{kc}",3.2,lambda x,kr=kr,kc=kc: np.tile(x,(kr,kc))))
            if all(ih==oh*kr and iw==ow*kc for (ih,iw),(oh,ow) in pairs):
                def down(x,kr=kr,kc=kc):
                    H,W=x.shape
                    if H%kr or W%kc:return None
                    y=np.zeros((H//kr,W//kc),np.int8)
                    for r in range(y.shape[0]):
                        for c in range(y.shape[1]):
                            block=x[r*kr:(r+1)*kr,c*kc:(c+1)*kc].ravel()
                            y[r,c]=Counter(map(int,block)).most_common(1)[0][0]
                    return y
                ps.append(Program(f"downmode_{kr}x{kc}",3.5,down))
    return ps

def base_programs(task):
    ps=[Program("id",1,lambda x:x.copy()),Program("r90",2,lambda x:np.rot90(x,1).copy()),Program("r180",2,lambda x:np.rot90(x,2).copy()),Program("r270",2,lambda x:np.rot90(x,3).copy()),Program("flip_lr",2,lambda x:np.fliplr(x).copy()),Program("flip_ud",2,lambda x:np.flipud(x).copy()),Program("transpose",2,lambda x:x.T.copy()),Program("crop_nonbg",2.5,crop_nonbg),Program("fill_holes",3,fill_holes),Program("bbox_fill",3,lambda x:bbox_draw(x,"fill")),Program("bbox_outline",3,lambda x:bbox_draw(x,"outline")),Program("complete_lr",3,lambda x:complete_sym(x,"lr")),Program("complete_ud",3,lambda x:complete_sym(x,"ud")),Program("complete_rot",3,lambda x:complete_sym(x,"rot"))]
    for sel in ["largest","smallest","top","bottom","left","right","most_holes","unique_size","unique_color"]:
        ps.append(Program(f"crop_obj_{sel}",3.1,lambda x,sel=sel: None if (o:=pick_obj(x,sel)) is None else render_obj(x,o,True)))
        ps.append(Program(f"keep_obj_{sel}",3.2,lambda x,sel=sel: None if (o:=pick_obj(x,sel)) is None else render_obj(x,o,False)))
    for op in ["union","inter","xor","a-b","b-a"]:
        ps.append(Program("panel_"+op,3.2,lambda x,op=op:panel_logic(x,op)))
    ps += scale_candidates(task)
    mapped=[]
    for p in list(ps):
        m=infer_global_color_map(task["train"],p)
        if m is not None:mapped.append(m)
    return ps+mapped


In [ ]:
# ---------- task-conditioned counterfactual probes ----------
def recolor_probe(x):
    x=A(x).copy(); b=bg(x)
    cols=[int(c) for c in np.unique(x) if int(c)!=b]
    if len(cols)<2:return None
    a,c=cols[0],cols[-1]
    y=x.copy(); y[x==a]=c; y[x==c]=a
    return y

def translate_probe(x):
    x=A(x); b=bg(x); pts=np.argwhere(x!=b)
    if not len(pts):return None
    for dr,dc in ((1,1),(-1,-1),(1,0),(0,1)):
        zz=np.full_like(x,b); ok=True
        for r,c in pts:
            rr,cc=r+dr,c+dc
            if not(0<=rr<x.shape[0] and 0<=cc<x.shape[1]):ok=False;break
            zz[rr,cc]=x[r,c]
        if ok:return zz
    return None

def isolate_probe(x,which):
    o=pick_obj(x,which)
    return None if o is None else render_obj(x,o,False)

def counterfactual_probes(train,max_probes=6):
    out=[]; seen=set(); funcs=[recolor_probe,translate_probe,lambda x:np.fliplr(A(x)).copy(),lambda x:np.flipud(A(x)).copy(),lambda x:isolate_probe(x,"largest"),lambda x:isolate_probe(x,"smallest")]
    for ex in train:
        x=A(ex["input"])
        for f in funcs:
            try:z=f(x)
            except:z=None
            if z is None:continue
            kz=K(z)
            if kz!=K(x) and kz not in seen:
                seen.add(kz);out.append(z)
                if len(out)>=max_probes:return out
    return out

class Meter:
    def __init__(self): self.candidate_input_evals=0
    def run(self,p,x): self.candidate_input_evals += 1; return p(x)

def fingerprint(p,inputs,meter):
    sig=[]
    for x in inputs:
        y=meter.run(p,x); sig.append(None if y is None else K(y))
    return tuple(sig)

def quotient(programs,inputs,meter,keep=2):
    groups=defaultdict(list)
    for p in programs: groups[fingerprint(p,inputs,meter)].append(p)
    reps=[]
    for g in groups.values(): reps.extend(sorted(g,key=lambda p:(p.cost,p.name))[:keep])
    return reps,len(groups)

def best_train_score(p,train):
    exact=0; px=[]
    for ex in train:
        y=p(ex["input"]); t=A(ex["output"]); ok=y is not None and y.shape==t.shape
        exact += int(ok and np.array_equal(y,t)); px.append(float(np.mean(y==t)) if ok else 0.)
    return exact,float(np.mean(px))


In [ ]:
# ---------- equal-depth A/B search ----------
def search_plain(task,max_depth=2,max_nodes=60000):
    base=base_programs(task); allp=[]; frontier=base; generated=0
    for d in range(1,max_depth+1):
        cand=frontier if d==1 else [compose(p,q) for p in frontier for q in base]
        if generated+len(cand)>max_nodes:cand=cand[:max(0,max_nodes-generated)]
        generated += len(cand); allp.extend(cand); frontier=cand
        if generated>=max_nodes:break
    meter=Meter(); fits=[]; best_exact=0;best_px=0.
    for p in allp:
        ex,px=0,[]; okall=True
        for e in task["train"]:
            y=meter.run(p,e["input"]); t=A(e["output"]); ok=y is not None and y.shape==t.shape
            ex+=int(ok and np.array_equal(y,t)); px.append(float(np.mean(y==t)) if ok else 0.); okall &= bool(ok and np.array_equal(y,t))
        best_exact=max(best_exact,ex);best_px=max(best_px,float(np.mean(px)))
        if okall:fits.append(p)
    return allp,fits,{"plain_generated":generated,"plain_candidate_input_evals":meter.candidate_input_evals,"plain_best_demo_exact":best_exact,"plain_best_pixel":best_px}

def search_quotient(task,max_depth=2,keep=2,use_probes=True,max_nodes=60000):
    base=base_programs(task); demos=[A(e["input"]) for e in task["train"]]; probes=counterfactual_probes(task["train"]) if use_probes else []; qinputs=demos+probes
    meter=Meter(); allreps=[]; frontier=base; generated=0; rows=[]
    for d in range(1,max_depth+1):
        cand=frontier if d==1 else [compose(p,q) for p in frontier for q in base]
        if generated+len(cand)>max_nodes:cand=cand[:max(0,max_nodes-generated)]
        generated+=len(cand); reps,ncls=quotient(cand,qinputs,meter,keep); allreps.extend(reps); allreps,_=quotient(allreps,qinputs,meter,keep)
        rows.append({"depth":d,"generated":len(cand),"classes":ncls,"kept":len(reps)}); frontier=reps
        if generated>=max_nodes:break
    fits=[];best_exact=0;best_px=0.
    for p in allreps:
        ex,px=0,[]; okall=True
        for e in task["train"]:
            y=meter.run(p,e["input"]);t=A(e["output"]);ok=y is not None and y.shape==t.shape
            ex+=int(ok and np.array_equal(y,t));px.append(float(np.mean(y==t)) if ok else 0.);okall &= bool(ok and np.array_equal(y,t))
        best_exact=max(best_exact,ex);best_px=max(best_px,float(np.mean(px)))
        if okall:fits.append(p)
    return allreps,fits,{"q_generated":generated,"q_candidate_input_evals":meter.candidate_input_evals,"q_final_reps":len(allreps),"q_best_demo_exact":best_exact,"q_best_pixel":best_px,"n_probes":len(probes),"depth_rows":rows}

def predict_from(programs,task):
    ranked=sorted(programs,key=lambda p:(-best_train_score(p,task["train"])[0],-best_train_score(p,task["train"])[1],p.cost,p.name)); out=[]
    for ex in task["test"]:
        uniq=[]
        for p in ranked:
            y=p(ex["input"])
            if y is None:continue
            ky=K(y)
            if ky not in {u[0] for u in uniq}:uniq.append((ky,y,p.name))
            if len(uniq)==2:break
        if not uniq:y=A(ex["input"]);uniq=[(K(y),y,"fallback")]
        if len(uniq)==1:uniq.append(uniq[0])
        out.append({"attempt_1":uniq[0][1].astype(int).tolist(),"attempt_2":uniq[1][1].astype(int).tolist()})
    return out


In [ ]:
# ---------- v2 experiment: held-out TRAIN tasks, not official evaluation ----------
MAX_DEPTH=2
KEEP=2
LIMIT=50
RUN_PLAIN_ABLATION=True

rows=[]
for i,tid in enumerate(HELD_IDS):
    if LIMIT is not None and i>=LIMIT:break
    task=TR[tid]; t0=time.perf_counter(); qpool,qfits,qd=search_quotient(task,MAX_DEPTH,KEEP,True)
    row={"task":tid,"outputs":len(task["test"]),"q_fit":int(bool(qfits)),**qd}
    if RUN_PLAIN_ABLATION:
        ppool,pfits,plain_diag=search_plain(task,MAX_DEPTH)
        row.update({"plain_fit":int(bool(pfits)),**plain_diag})
        row["generated_reduction"]=plain_diag["plain_generated"]/max(1,qd["q_generated"])
        row["eval_reduction"]=plain_diag["plain_candidate_input_evals"]/max(1,qd["q_candidate_input_evals"])
        row["fit_preserved"]=int((not pfits) or bool(qfits))
    row["seconds"]=time.perf_counter()-t0; rows.append(row)

# re-import defensively so a local diagnostics variable cannot shadow pandas
import pandas as pd
df=pd.DataFrame(rows)
print("tasks:",len(df)); print("quotient coverage:",df.q_fit.mean())
if RUN_PLAIN_ABLATION:
    print("plain coverage:",df.plain_fit.mean()); print("fit preservation:",df.fit_preserved.mean()); print("median structural generation reduction:",df.generated_reduction.median()); print("median candidate-input eval reduction:",df.eval_reduction.median())
print("median best demo exact pairs:",df.q_best_demo_exact.median()); print("median best pixel agreement:",df.q_best_pixel.median())
display(df.head()); display(df.describe(include="all")); df.to_csv("object_relation_v2_heldout.csv",index=False)


In [ ]:
RUN_OFFICIAL_EVAL=False
if RUN_OFFICIAL_EVAL:
    erows=[];correct=total=0
    for tid,task in EV.items():
        pool,fits,d=search_quotient(task,MAX_DEPTH,KEEP,True); source=fits if fits else pool; pred=predict_from(source,task); hit=0
        for p,y in zip(pred,ES[tid]):
            y=A(y);a=A(p["attempt_1"]);b=A(p["attempt_2"]);ok=(a.shape==y.shape and np.array_equal(a,y)) or (b.shape==y.shape and np.array_equal(b,y));hit+=int(ok);correct+=int(ok);total+=1
        erows.append({"task":tid,"correct":hit,"outputs":len(ES[tid]),"fit":int(bool(fits)),**d})
    edf=pd.DataFrame(erows); print("official eval exact output accuracy:",correct/max(1,total)); print("official eval coverage:",edf.fit.mean()); edf.to_csv("object_relation_v2_official_eval.csv",index=False)


In [ ]:
WRITE_SUBMISSION=False
if WRITE_SUBMISSION:
    submission={};srows=[]
    for tid,task in TE.items():
        pool,fits,d=search_quotient(task,MAX_DEPTH,KEEP,True); source=fits if fits else pool; submission[tid]=predict_from(source,task); srows.append({"task":tid,"fit":int(bool(fits)),**d})
    assert set(submission)==set(TE)
    for tid,v in submission.items():
        assert len(v)==len(TE[tid]["test"]); assert all(set(o)=={"attempt_1","attempt_2"} for o in v)
    json.dump(submission,open("submission.json","w")); pd.DataFrame(srows).to_csv("object_relation_v2_test.csv",index=False); print("wrote submission.json for",len(submission),"tasks")


## Decision rule for v2

Do **not** optimize ranking if `q_fit` remains near zero.

Continue this direction only if the held-out training split shows at least one of:

* materially higher exact-demonstration coverage than v1;
* equal coverage with fewer candidate generations / candidate-input executions;
* probe quotienting preserves fitting programs while demo-only pruning does not.

If coverage stays very low, the next representation should become more relational: explicit object graphs, learned selectors/relations, and output construction from object sets rather than deeper compositions of grid functions.
